In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch transformers datasets peft bitsandbytes py_vncorenlp \
    accelerate trl beir \
    numpy pandas scikit-learn sentencepiece tokenizers tqdm underthesea

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 21.1 MB/s eta 0:00:00
  Created wheel for py_vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=cf250b2815ab3f069d6292eea3392e596394972f17629bba2ceca4c515bceed0
  Stored in directory: /root/.cache/pip/wheels/db/e5/ff/f4a1b4ece36e8582db1ca71150a34e987e65df50c35974e9bb
Successfully built py_vncorenlp


In [ ]:
import os

def install_java():
  !apt update -qq
  !apt-get install -y openjdk-21-jdk-headless -qq > /dev/null
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
  !java -version

install_java()

54 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "21.0.9" 2025-10-21
OpenJDK Runtime Environment (build 21.0.9+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.9+10-Ubuntu-122.04, mixed mode, sharing)


In [ ]:
!git clone https://github.com/Tommachilez/improving-learned-index.git
%cd /content/improving-learned-index

Cloning into 'improving-learned-index'...
remote: Enumerating objects: 1266, done.
remote: Counting objects: 100% (362/362), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 1266 (delta 314), reused 300 (delta 274), pack-reused 904 (from 1)
Receiving objects: 100% (1266/1266), 211.43 KiB | 9.61 MiB/s, done.
Resolving deltas: 100% (926/926), done.
/content/improving-learned-index


In [ ]:
dataset_path = "/content/drive/MyDrive/KLTN_Project/Datasets"
best_passage = f"{dataset_path}/viranker_maxp/best_passage.csv"
doc_mapping = f"{dataset_path}/viranker_maxp/best_passage_mapping.csv"
pretok_doc = f"{dataset_path}/viranker_maxp/best_passage_pretokenized.csv"
query_mapping = f"{dataset_path}/updated_unique_query_mapping.csv"
pretokenized_queries = f"{dataset_path}/vn_mining/queries_pretokenized.jsonl"
output_queries = f"{dataset_path}/deeperimpact_vifc_maxp/train_queries.tsv"
output_docs = f"{dataset_path}/deeperimpact_vifc_maxp/gemini_expanded_documents.tsv"
output_expansions = f"{dataset_path}/deeperimpact_vifc_maxp/expansions.csv"
vncorenlp_path = "/content/drive/MyDrive/KLTN_Project/Models/vncorenlp_models"
stopwords_path = f"{dataset_path}/stopwords-vi.txt"

# Merge expansions

In [ ]:
!python -m src.deep_impact.scripts.create_unique_passage_mapping \
    --input_csv {best_passage} \
    --output_csv {doc_mapping}

Processing /content/drive/MyDrive/KLTN_Project/Datasets/viranker_maxp/best_passage.csv -> /content/drive/MyDrive/KLTN_Project/Datasets/viranker_maxp/best_passage_mapping.csv ...
Filtering Unique Passages: 992229it [00:26, 37745.01it/s]

Processing Complete.
Total Rows Read:   992229
Unique Passages:   10026
Output saved to:   /content/drive/MyDrive/KLTN_Project/Datasets/viranker_maxp/best_passage_mapping.csv


In [ ]:
!python -m src.deep_impact.scripts.preprocess_passages \
    --input_csv {doc_mapping} \
    --output_csv {pretok_doc} \
    --vncorenlp_path {vncorenlp_path} \
    --stopwords_path {stopwords_path}

>>> Initializing VietnameseProcessor...
2026-01-02 04:15:10 INFO  WordSegmenter:24 - Loading Word Segmentation model
>>> Processing /content/drive/MyDrive/KLTN_Project/Datasets/viranker_maxp/best_passage_mapping.csv -> /content/drive/MyDrive/KLTN_Project/Datasets/viranker_maxp/best_passage_pretokenized.csv
Tokenizing Passages: 10026it [01:06, 151.34it/s]
>>> Done. Processed 10026 passages.


In [ ]:
!python -m src.deep_impact.scripts.create_training_files_maxp \
    --best_passages {pretok_doc} \
    --query_mapping {query_mapping} \
    --pretokenized_queries {pretokenized_queries} \
    --output_queries_tsv {output_queries} \
    --output_docs_tsv {output_docs} \
    --output_expansion_csv {output_expansions}

Converting /content/drive/MyDrive/KLTN_Project/Datasets/updated_unique_query_mapping.csv to /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact_vifc_maxp/train_queries.tsv...
Processing Queries: 136706it [00:02, 65267.81it/s] 
Aggregating query terms from /content/drive/MyDrive/KLTN_Project/Datasets/vn_mining/queries_pretokenized.jsonl...
Reading Queries JSONL: 61931it [00:02, 30159.48it/s]
Loading tokenizer: xlm-roberta-base...
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 167kB/s]
config.json: 100% 615/615 [00:00<00:00, 4.79MB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 7.93MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:01<00:00, 8.07MB/s]
Building collection from /content/drive/MyDrive/KLTN_Project/Datasets/viranker_maxp/best_passage_pretokenized.csv...
- Expanded Docs TSV: /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact_vifc_maxp/gemini_expanded_documents.tsv
- Expansion Terms CSV: /content/drive/MyDrive/KLTN_Project/Datasets/deeperimpact_vifc_maxp/

# Trim cross-encoder scores file


In [ ]:
!python -m src.deep_impact.scripts.trim_scores \
    --collection_path /content/drive/MyDrive/KLTN_Project/Datasets/expanded_collection.tsv \
    --scores_path /content/drive/MyDrive/KLTN_Project/Datasets/MS_MARCO/cross-encoder-ms-marco-MiniLM-L-6-v2-scores.pkl.gz \
    --output_path /content/drive/MyDrive/KLTN_Project/Datasets/MS_MARCO/scores.trimmed.pkl.gz

Loading valid PIDs from collection: /content/drive/MyDrive/KLTN_Project/Datasets/expanded_collection.tsv...
Reading collection: 39072it [00:02, 18962.28it/s]
Found 39072 unique valid PIDs in 39072 lines.
Loading scores data from /content/drive/MyDrive/KLTN_Project/Datasets/MS_MARCO/cross-encoder-ms-marco-MiniLM-L-6-v2-scores.pkl.gz...
Scores data loaded. Found scores for 808731 QIDs.
Trimming scores data...
Trimming QIDs: 100% 808731/808731 [00:14<00:00, 55574.77it/s]

--- Trimming Stats ---
Original PID-Score entries:  160,088,133
 Trimmed PID-Score entries:      707,604
   Removed entries (missing):  159,380,529

Saving trimmed data to /content/drive/MyDrive/KLTN_Project/Datasets/MS_MARCO/scores.trimmed.pkl.gz...
Save complete.

Done. Your new scores file is ready for training.


# Train

In [ ]:
scores = f"{dataset_path}/viranker_maxp/scores.pkl.gz"

In [ ]:
!torchrun --standalone --nproc_per_node=gpu -m src.deep_impact.train \
  --dataset_path {scores} \
  --queries_path {output_queries} \
  --collection_path {output_docs} \
  --checkpoint_dir /content/drive/MyDrive/KLTN_Project/Models/deeperimpact_xlmr_vifc_maxp\
  --xlmr \
  --batch_size 4 \
  --save_every 5000 \
  --lr 1e-6 \
  --gradient_accumulation_steps 8 \
  --distil_kl \
  --no_beir_eval \
  --use_wandb \
  --vncorenlp_path {vncorenlp_path} \
#   --qrels_path /content/drive/MyDrive/KLTN_Project/Datasets/MS_MARCO/qrels.train \


2026-01-02 04:18:29.709766: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767327509.730644    2903 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767327509.736990    2903 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767327509.752745    2903 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767327509.752772    2903 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767327509.752776    2903 computation_placer.cc:177] computation placer alr